# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulah-naeem/FlyRank-ml-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1:** "Pages that were updated in the last 6 months show a 30% higher ranking retention compared to stale pages."
**My methodology question:** How is 'updated' defined and recorded? Does a minor meta-tag tweak count as an update, or is there a threshold for meaningful content addition? If minor tweaks count, are we just measuring SEO activity rather than content quality?

**Finding 2:** "Heuristic health scores strongly correlate with long-term organic traffic growth."
**My methodology question:** Are the health scores computed using historical traffic data? If the score incorporates past traffic trends, then predicting future traffic growth from it might just be autocorrelation rather than the health score discovering a true underlying quality signal.

In [1]:
# No code needed for this section.

## 2. My model under an honest split (before/after)

Here we compare a naive random split (which leaks client-specific baseline behaviors) against an honest grouped split (GroupShuffleSplit by `client_id`), where the model must predict on entirely unseen clients.

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Load Data
df = pd.read_csv('../../data/processed/refresh_feature_vector.csv')

# Define features and target
drop_cols = ['content_id', 'client_id', 'is_declining_label', 'trend_direction', 'trend_pct', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d']
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
features = [c for c in numeric_cols if c not in drop_cols]
target = 'is_declining_label'

def evaluate_split(X_train, y_train, X_test, y_test, name):
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('model', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1))
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict_proba(X_test)[:, 1]
    
    # Precision@50
    results_df = pd.DataFrame({'score': preds, 'target': y_test})
    p50 = results_df.sort_values('score', ascending=False).head(50)['target'].mean()
    print(f"{name} Split - Precision@50: {p50:.3f} | Base Rate: {y_test.mean():.3f}")
    return pipeline

print("=== BEFORE: Naive Random Split (Leaky) ===")
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(df[features], df[target], test_size=0.2, random_state=42)
evaluate_split(X_train_rnd, y_train_rnd, X_test_rnd, y_test_rnd, "Random")

print("\n=== AFTER: Honest Grouped Split (by client_id) ===")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df[target], groups=df['client_id']))
X_train_grp, y_train_grp = df.iloc[train_idx][features], df.iloc[train_idx][target]
X_test_grp, y_test_grp = df.iloc[test_idx][features], df.iloc[test_idx][target]
pipeline_grp = evaluate_split(X_train_grp, y_train_grp, X_test_grp, y_test_grp, "Grouped")


=== BEFORE: Naive Random Split (Leaky) ===
Random Split - Precision@50: 1.000 | Base Rate: 0.545

=== AFTER: Honest Grouped Split (by client_id) ===
Grouped Split - Precision@50: 0.960 | Base Rate: 0.511


## 3. Leakage audit

We investigate if any single feature is suspiciously dominant (a classic sign of label leakage). We also intentionally inject a leaky feature to verify our harness detects it.

In [3]:
# 1. Feature Importance Check on the Honest Model
feature_names = features + [f"{f}_missing" for i, f in enumerate(features) if i in pipeline_grp.named_steps['imputer'].indicator_.features_]
importances = pipeline_grp.named_steps['model'].feature_importances_

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
print("Top 5 Features in Honest Model:")
print(imp_df.head(5).to_string(index=False))
print("\nVerdict: No single feature dominates with >0.5 importance. Features look legitimate.")

# 2. The Verification Attack (Injecting Leakage)
print("\n=== INJECTING A LEAKY FEATURE ===")
df_leaky = df.copy()
# Injecting a feature heavily correlated with the label
df_leaky['leaky_future_flag'] = df_leaky[target] * 0.9 + np.random.normal(0, 0.1, len(df_leaky))
leaky_features = features + ['leaky_future_flag']

X_train_lk, y_train_lk = df_leaky.iloc[train_idx][leaky_features], df_leaky.iloc[train_idx][target]
X_test_lk, y_test_lk = df_leaky.iloc[test_idx][leaky_features], df_leaky.iloc[test_idx][target]

pipeline_leaky = evaluate_split(X_train_lk, y_train_lk, X_test_lk, y_test_lk, "Leaky Grouped")
leaky_importances = pipeline_leaky.named_steps['model'].feature_importances_
leaky_feature_names = leaky_features + [f"{f}_missing" for i, f in enumerate(leaky_features) if i in pipeline_leaky.named_steps['imputer'].indicator_.features_]
imp_leaky_df = pd.DataFrame({'feature': leaky_feature_names, 'importance': leaky_importances}).sort_values('importance', ascending=False)

print("\nTop 3 Features in Leaky Model:")
print(imp_leaky_df.head(3).to_string(index=False))
print("\nVerdict: The harness correctly catches the injected leaky feature (it instantly dominated importance). Harness is reliable.")


Top 5 Features in Honest Model:
              feature  importance
 impressions_prev_30d    0.269988
      impressions_90d    0.110538
days_with_impressions    0.102313
         avg_position    0.086473
 impressions_last_30d    0.069171

Verdict: No single feature dominates with >0.5 importance. Features look legitimate.

=== INJECTING A LEAKY FEATURE ===
Leaky Grouped Split - Precision@50: 1.000 | Base Rate: 0.511

Top 3 Features in Leaky Model:
              feature  importance
    leaky_future_flag    0.693583
 impressions_prev_30d    0.091020
days_with_impressions    0.041967

Verdict: The harness correctly catches the injected leaky feature (it instantly dominated importance). Harness is reliable.


## 4. Claim rewrite

**Original bold claim:** "Our ML model perfectly predicts Google's algorithm and exactly identifies which pages are decaying with 96% precision."

**Safe rewrite:** "Our model observed directional indicators of content decay. In our validation split, it demonstrated strong decision-support capabilities, prioritizing decaying pages with 96% precision in the top 50 recommendations, compared to a 44% rule-based baseline."

In [4]:
# No code needed for this section.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.